# 0. Loading in


In [ ]:

from pathlib import Path


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid')

DATA_DIR = Path('.') / 'data'



In [9]:
full_spine = pd.read_csv(DATA_DIR/'made/AA_MASTER_V8.csv')

C:\Users\danie\AppData\Local\Temp\ipykernel_9240\448201957.py:1: DtypeWarning: Columns (15,16,20,21,34,35,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  full_spine = pd.read_csv(DATA_DIR/'made/AA_MASTER_V8.csv')


# 1. Add travel distance

In [10]:


def add_haversine_distance(
    df: pd.DataFrame,
    orig_lat: str = "ORG_LAT",
    orig_lon: str = "ORG_LNGTD",
    dest_lat: str = "DES_LAT",
    dest_lon: str = "DES_LNGTD",
) -> pd.DataFrame:
    """Computes great-circle distance in miles (and log1p miles) between Origin and Destination."""
    lat1 = np.radians(pd.to_numeric(df[orig_lat], errors="coerce"))
    lon1 = np.radians(pd.to_numeric(df[orig_lon], errors="coerce"))
    lat2 = np.radians(pd.to_numeric(df[dest_lat], errors="coerce"))
    lon2 = np.radians(pd.to_numeric(df[dest_lon], errors="coerce"))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

    earth_radius_miles = 3958.7613
    df["Route_Distance_Miles"] = earth_radius_miles * c
    df["Route_Distance_log1p"] = np.log1p(df["Route_Distance_Miles"])
    return df

full_spine = add_haversine_distance(full_spine)


In [11]:
full_spine.head()

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,MPAX_L24,PAX_SHARE_L24,AIRLN_COUNT_L24,ORG_CARGO_WGT,ORG_CARGO_SL,ORG_CARGO_HUB,DES_CARGO_WGT,DES_CARGO_SL,DES_CARGO_HUB,ORG_ENP_SL,ORG_ENP_HUB,ORG_ENPLANEMENT,DES_ENP_SL,DES_ENP_HUB,...,DES_CNTRY_CD,DES_LNGST_RUNWAY_FT_QTY,DES_ELEVATN_FT_QTY,DES_LAT,DES_LNGTD,DES_CNTY_FIPS,DES_CBSA_CD,ORG_CNTY_GDP_LOG,ORG_CNTY_INC,DES_CNTY_GDP_LOG,DES_CNTY_INC,ORG_CNTRY_GDP,DES_CNTRY_GDP,ORG_CBSA_POP,ORG_CBSA_POP_GRWT,ORG_CBSA_NET_MIG,DES_CBSA_POP,DES_CBSA_POP_GRWT,DES_CBSA_NET_MIG,MIG_OUT_LOG,MIG_IN_LOG,MIG_GROSS_LOG,MIG_NET_LOG,Route_Distance_Miles,Route_Distance_log1p
0,ABQ,ALB,2019,1,ABQ-ALB,11.07,5.54,8,2,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,USA,8500,285,42.75,-73.80,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,"1,829.80",7.51
1,ABQ,ALB,2019,2,ABQ-ALB,10.83,5.21,8,1,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,USA,8500,285,42.75,-73.80,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,"1,829.80",7.51
2,ABQ,ALB,2019,3,ABQ-ALB,11.71,5.96,10,0,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,USA,8500,285,42.75,-73.80,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,"1,829.80",7.51
3,ABQ,ALB,2019,4,ABQ-ALB,11.63,6.11,8,0,1,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,USA,8500,285,42.75,-73.80,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,"1,829.80",7.51
4,ABQ,ALB,2019,5,ABQ-ALB,11.91,6.37,9,1,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,USA,8500,285,42.75,-73.80,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,"1,829.80",7.51


In [12]:
full_spine.drop(columns=['Route_Distance_Miles'],inplace=True)
full_spine.rename(columns={'Route_Distance_log1p':'DIST_LOG'},inplace=True)
full_spine.head()

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,MPAX_L24,PAX_SHARE_L24,AIRLN_COUNT_L24,ORG_CARGO_WGT,ORG_CARGO_SL,ORG_CARGO_HUB,DES_CARGO_WGT,DES_CARGO_SL,DES_CARGO_HUB,ORG_ENP_SL,ORG_ENP_HUB,ORG_ENPLANEMENT,DES_ENP_SL,DES_ENP_HUB,...,ORG_CBSA_CD,DES_CNTRY_CD,DES_LNGST_RUNWAY_FT_QTY,DES_ELEVATN_FT_QTY,DES_LAT,DES_LNGTD,DES_CNTY_FIPS,DES_CBSA_CD,ORG_CNTY_GDP_LOG,ORG_CNTY_INC,DES_CNTY_GDP_LOG,DES_CNTY_INC,ORG_CNTRY_GDP,DES_CNTRY_GDP,ORG_CBSA_POP,ORG_CBSA_POP_GRWT,ORG_CBSA_NET_MIG,DES_CBSA_POP,DES_CBSA_POP_GRWT,DES_CBSA_NET_MIG,MIG_OUT_LOG,MIG_IN_LOG,MIG_GROSS_LOG,MIG_NET_LOG,DIST_LOG
0,ABQ,ALB,2019,1,ABQ-ALB,11.07,5.54,8,2,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,10740,USA,8500,285,42.75,-73.80,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,7.51
1,ABQ,ALB,2019,2,ABQ-ALB,10.83,5.21,8,1,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,10740,USA,8500,285,42.75,-73.80,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,7.51
2,ABQ,ALB,2019,3,ABQ-ALB,11.71,5.96,10,0,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,10740,USA,8500,285,42.75,-73.80,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,7.51
3,ABQ,ALB,2019,4,ABQ-ALB,11.63,6.11,8,0,1,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,10740,USA,8500,285,42.75,-73.80,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,7.51
4,ABQ,ALB,2019,5,ABQ-ALB,11.91,6.37,9,1,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,10740,USA,8500,285,42.75,-73.80,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,7.51


In [13]:
full_spine.drop(columns=['ORG_LAT','ORG_LNGTD','DES_LAT','DES_LNGTD'],inplace=True)
full_spine.head()

,ORG_AIRPRT,DES_AIRPRT,DEP_YEAR,DEP_MONTH,ROUTE,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,MPAX_L24,PAX_SHARE_L24,AIRLN_COUNT_L24,ORG_CARGO_WGT,ORG_CARGO_SL,ORG_CARGO_HUB,DES_CARGO_WGT,DES_CARGO_SL,DES_CARGO_HUB,ORG_ENP_SL,ORG_ENP_HUB,ORG_ENPLANEMENT,DES_ENP_SL,DES_ENP_HUB,...,ORG_ELEVATN_FT_QTY,ORG_CNTY_FIPS,ORG_CBSA_CD,DES_CNTRY_CD,DES_LNGST_RUNWAY_FT_QTY,DES_ELEVATN_FT_QTY,DES_CNTY_FIPS,DES_CBSA_CD,ORG_CNTY_GDP_LOG,ORG_CNTY_INC,DES_CNTY_GDP_LOG,DES_CNTY_INC,ORG_CNTRY_GDP,DES_CNTRY_GDP,ORG_CBSA_POP,ORG_CBSA_POP_GRWT,ORG_CBSA_NET_MIG,DES_CBSA_POP,DES_CBSA_POP_GRWT,DES_CBSA_NET_MIG,MIG_OUT_LOG,MIG_IN_LOG,MIG_GROSS_LOG,MIG_NET_LOG,DIST_LOG
0,ABQ,ALB,2019,1,ABQ-ALB,11.07,5.54,8,2,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,5355,35001,10740,USA,8500,285,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,7.51
1,ABQ,ALB,2019,2,ABQ-ALB,10.83,5.21,8,1,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,5355,35001,10740,USA,8500,285,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,7.51
2,ABQ,ALB,2019,3,ABQ-ALB,11.71,5.96,10,0,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,5355,35001,10740,USA,8500,285,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,7.51
3,ABQ,ALB,2019,4,ABQ-ALB,11.63,6.11,8,0,1,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,5355,35001,10740,USA,8500,285,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,7.51
4,ABQ,ALB,2019,5,ABQ-ALB,11.91,6.37,9,1,0,0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,P,M,"2,412,328.00",P,S,...,5355,35001,10740,USA,8500,285,36001,10580,24.32,"41,936.00",24.18,"56,786.00",11.00,11.00,13.73,0.00,0.00,13.71,0.01,-0.00,0.00,0.00,0.00,0.00,7.51


In [15]:
full_spine.columns

Index(['ORG_AIRPRT', 'DES_AIRPRT', 'DEP_YEAR', 'DEP_MONTH', 'ROUTE',
       'OD_REV_USD_LOG', 'OD_PAX_LOG', 'WEEKEND', 'HOLIDAY', 'EASTER', 'COVID',
       'MPAX_L24', 'PAX_SHARE_L24', 'AIRLN_COUNT_L24', 'ORG_CARGO_WGT',
       'ORG_CARGO_SL', 'ORG_CARGO_HUB', 'DES_CARGO_WGT', 'DES_CARGO_SL',
       'DES_CARGO_HUB', 'ORG_ENP_SL', 'ORG_ENP_HUB', 'ORG_ENPLANEMENT',
       'DES_ENP_SL', 'DES_ENP_HUB', 'DES_ENPLANEMENT', 'OD_PAX_LOG_L24',
       'OD_REV_USD_LOG_L24', 'L24_COVID', 'ORG_CNTRY_CD',
       'ORG_LNGST_RUNWAY_FT_QTY', 'ORG_ELEVATN_FT_QTY', 'ORG_CNTY_FIPS',
       'ORG_CBSA_CD', 'DES_CNTRY_CD', 'DES_LNGST_RUNWAY_FT_QTY',
       'DES_ELEVATN_FT_QTY', 'DES_CNTY_FIPS', 'DES_CBSA_CD',
       'ORG_CNTY_GDP_LOG', 'ORG_CNTY_INC', 'DES_CNTY_GDP_LOG', 'DES_CNTY_INC',
       'ORG_CNTRY_GDP', 'DES_CNTRY_GDP', 'ORG_CBSA_POP', 'ORG_CBSA_POP_GRWT',
       'ORG_CBSA_NET_MIG', 'DES_CBSA_POP', 'DES_CBSA_POP_GRWT',
       'DES_CBSA_NET_MIG', 'MIG_OUT_LOG', 'MIG_IN_LOG', 'MIG_GROSS_LOG',
       'MI

In [17]:
full_spine['ORG_CNTY_INC_LOG'] = np.log1p(full_spine['ORG_CNTY_INC'])
full_spine['DES_CNTY_INC_LOG'] = np.log1p(full_spine['DES_CNTY_INC'])

# 2. International tags + Econ Score

In [ ]:


def add_route_type_and_gravity(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds:
      1. `route_type` (0 = US Domestic, 1 = US-International, 2 = Foreign-Foreign)
      2. `is_international` (0 = Domestic US, 1 = Any International)
      3. `Economic_Gravity_log1p` (Unified log-space gravity score using County GDP
         for US endpoints and Country GDP per capita fallback for international endpoints)
      4. `Income_Power_log1p` (Combined purchasing power at both endpoints)
    """
    df = df.copy()

    # 1. Route Type Flag (0 = US-US, 1 = US-Intl, 2 = Foreign-Foreign)
    orig_is_us = (df["ORG_CNTRY_CD"] == "USA").astype(int)
    dest_is_us = (df["DES_CNTRY_CD"] == "USA").astype(int)

    # Number of non-US stops on the route (0, 1, or 2)
    df["route_type"] = (1 - orig_is_us) + (1 - dest_is_us)
    df["INTRNAT"] = (df["route_type"] > 0).astype(int)

    # 2. Unified Economic Mass per Endpoint
    #    Use County Real GDP where available (US counties); fall back to Country GDP per capita
    #    for international endpoints (and territories like SJU/GUM) where County GDP is NaN
    orig_econ_mass = df["ORG_CNTY_GDP_LOG"].fillna(df["ORG_CNTRY_GDP"])
    dest_econ_mass = df["DES_CNTY_GDP_LOG"].fillna(df["DES_CNTRY_GDP"])

    # 3. Economic Gravity Score becomes addition in log space
    dist_log = df["DIST_LOG"].replace(0, np.nan)
    df["ECON_LOG"] = (orig_econ_mass + dest_econ_mass) - dist_log

    # 4. Pure Market Scale Interaction (Sum of both endpoints' economic mass without distance penalty)
    df["ECON_SUM_LOG"] = orig_econ_mass + dest_econ_mass

    # 5. Consumer Purchasing Power Score (Per Capita Income for US, Country GDP pc fallback for Intl)
    orig_purchasing = df["ORG_CNTY_INC_LOG"].fillna(df["ORG_CNTRY_GDP"])
    dest_purchasing = df["DES_CNTY_INC_LOG"].fillna(df["DES_CNTRY_GDP"])
    df["PURCHASE_PWR_LOG"] = orig_purchasing + dest_purchasing

    return df

full_spine = add_route_type_and_gravity(full_spine)


In [19]:
full_spine.columns

Index(['ORG_AIRPRT', 'DES_AIRPRT', 'DEP_YEAR', 'DEP_MONTH', 'ROUTE',
       'OD_REV_USD_LOG', 'OD_PAX_LOG', 'WEEKEND', 'HOLIDAY', 'EASTER', 'COVID',
       'MPAX_L24', 'PAX_SHARE_L24', 'AIRLN_COUNT_L24', 'ORG_CARGO_WGT',
       'ORG_CARGO_SL', 'ORG_CARGO_HUB', 'DES_CARGO_WGT', 'DES_CARGO_SL',
       'DES_CARGO_HUB', 'ORG_ENP_SL', 'ORG_ENP_HUB', 'ORG_ENPLANEMENT',
       'DES_ENP_SL', 'DES_ENP_HUB', 'DES_ENPLANEMENT', 'OD_PAX_LOG_L24',
       'OD_REV_USD_LOG_L24', 'L24_COVID', 'ORG_CNTRY_CD',
       'ORG_LNGST_RUNWAY_FT_QTY', 'ORG_ELEVATN_FT_QTY', 'ORG_CNTY_FIPS',
       'ORG_CBSA_CD', 'DES_CNTRY_CD', 'DES_LNGST_RUNWAY_FT_QTY',
       'DES_ELEVATN_FT_QTY', 'DES_CNTY_FIPS', 'DES_CBSA_CD',
       'ORG_CNTY_GDP_LOG', 'ORG_CNTY_INC', 'DES_CNTY_GDP_LOG', 'DES_CNTY_INC',
       'ORG_CNTRY_GDP', 'DES_CNTRY_GDP', 'ORG_CBSA_POP', 'ORG_CBSA_POP_GRWT',
       'ORG_CBSA_NET_MIG', 'DES_CBSA_POP', 'DES_CBSA_POP_GRWT',
       'DES_CBSA_NET_MIG', 'MIG_OUT_LOG', 'MIG_IN_LOG', 'MIG_GROSS_LOG',
       'MI

In [20]:
full_spine.isna().mean()

ORG_AIRPRT         0.00
DES_AIRPRT         0.00
DEP_YEAR           0.00
DEP_MONTH          0.00
ROUTE              0.00
                   ... 
route_type         0.00
INTRNAT            0.00
ECON_LOG           0.03
ECON_SUM_LOG       0.03
PURCHASE_PWR_LOG   0.03
Length: 63, dtype: float64

In [21]:
full_spine.INTRNAT.value_counts()

INTRNAT
0    1045350
1     923335
Name: count, dtype: int64

In [22]:
print("Route Type Breakdown (0=US-US, 1=US-Intl, 2=Intl-Intl):")
print(full_spine["route_type"].value_counts(dropna=False))

print("\nGravity Score Null %:", (full_spine["ECON_LOG"].isna().mean() * 100).round(2))
print(full_spine[["ROUTE", "route_type", "DIST_LOG", "ECON_LOG"]].drop_duplicates("ROUTE").head(6))


Route Type Breakdown (0=US-US, 1=US-Intl, 2=Intl-Intl):
route_type
0    1045350
1     923335
Name: count, dtype: int64

Gravity Score Null %: 2.98
       ROUTE  route_type  DIST_LOG  ECON_LOG
0    ABQ-ALB           0      7.51     40.99
115  ABQ-AMS           1      8.54     26.59
230  ABQ-ANC           0      7.87     40.32
345  ABQ-ATH           1      8.78     25.38
460  ABQ-ATL           0      7.14     40.79
575  ABQ-AUA           1      7.92     26.65


In [28]:
full_spine.columns

Index(['ORG_AIRPRT', 'DES_AIRPRT', 'DEP_YEAR', 'DEP_MONTH', 'ROUTE',
       'OD_REV_USD_LOG', 'OD_PAX_LOG', 'WEEKEND', 'HOLIDAY', 'EASTER', 'COVID',
       'MPAX_L24', 'PAX_SHARE_L24', 'AIRLN_COUNT_L24', 'ORG_CARGO_WGT',
       'ORG_CARGO_SL', 'ORG_CARGO_HUB', 'DES_CARGO_WGT', 'DES_CARGO_SL',
       'DES_CARGO_HUB', 'ORG_ENP_SL', 'ORG_ENP_HUB', 'ORG_ENPLANEMENT',
       'DES_ENP_SL', 'DES_ENP_HUB', 'DES_ENPLANEMENT', 'OD_PAX_LOG_L24',
       'OD_REV_USD_LOG_L24', 'L24_COVID', 'ORG_CNTRY_CD',
       'ORG_LNGST_RUNWAY_FT_QTY', 'ORG_ELEVATN_FT_QTY', 'ORG_CNTY_FIPS',
       'ORG_CBSA_CD', 'DES_CNTRY_CD', 'DES_LNGST_RUNWAY_FT_QTY',
       'DES_ELEVATN_FT_QTY', 'DES_CNTY_FIPS', 'DES_CBSA_CD',
       'ORG_CNTY_GDP_LOG', 'ORG_CNTY_INC', 'DES_CNTY_GDP_LOG', 'DES_CNTY_INC',
       'ORG_CNTRY_GDP', 'DES_CNTRY_GDP', 'ORG_CBSA_POP', 'ORG_CBSA_POP_GRWT',
       'ORG_CBSA_NET_MIG', 'DES_CBSA_POP', 'DES_CBSA_POP_GRWT',
       'DES_CBSA_NET_MIG', 'MIG_OUT_LOG', 'MIG_IN_LOG', 'MIG_GROSS_LOG',
       'MI

In [ ]:


def engineer_final_features(
    df: pd.DataFrame,
    pax_col: str = "OD_PAX_LOG",
    rev_col: str = "OD_REV_USD_LOG",
    drop_covid_in_profile: bool = True,
) -> pd.DataFrame:
    """
    Adds leak-free domain features to `full_spine`:
      1. Frozen Route-Month Seasonality Profiles (computed strictly <= cutoff T)
      2. Lag-24 Implied Average Ticket Yield (log1p fare per passenger)
      3. Cyclical Month (sin/cos) + Peak Travel Season Flags
      4. Combined Endpoint Cargo & Enplanement Hub Scale
      5. Long-Haul Intercontinental Widebody Flag
    """
    df = df.copy()

    
    # Auto-detect exact historical cutoff T last month where target is not NaN
    hist_mask = df[pax_col].notna()
    if drop_covid_in_profile:
        hist_mask = hist_mask & (~df["DEP_YEAR"].isin([2020, 2021]))

    hist_only = df.loc[hist_mask, ["ROUTE", "DEP_MONTH", pax_col, rev_col]].copy()

    # Convert back from log1p -> raw scale before averaging, then re-apply log1p
    hist_only["_RAW_PAX"] = np.expm1(hist_only[pax_col].clip(lower=0))
    hist_only["_RAW_REV"] = np.expm1(hist_only[rev_col].clip(lower=0))

    route_month_profile = (
        hist_only.groupby(["ROUTE", "DEP_MONTH"], as_index=False)
        .agg(
            _mean_pax=("_RAW_PAX", "mean"),
            _mean_rev=("_RAW_REV", "mean"),
            Route_Active_Months_Hist=("_RAW_PAX", lambda s: (s > 0).sum()),
        )
    )
    route_month_profile["Frozen_Route_Month_Pax_log1p"] = np.log1p(route_month_profile["_mean_pax"])
    route_month_profile["Frozen_Route_Month_Rev_log1p"] = np.log1p(route_month_profile["_mean_rev"])

    df = df.merge(
        route_month_profile[
            ["ROUTE", "DEP_MONTH", "Frozen_Route_Month_Pax_log1p", "Frozen_Route_Month_Rev_log1p", "Route_Active_Months_Hist"]
        ],
        on=["ROUTE", "DEP_MONTH"],
        how="left",
    )

    pax_l24 = f"{pax_col}_L24"
    rev_l24 = f"{rev_col}_L24"
    if pax_l24 in df.columns and rev_l24 in df.columns:
        # Only valid when Passengers_lag_24 > 0
        df["AVG_FARE_LOG_L24"] = np.where(
            df[pax_l24] > 0,
            (df[rev_l24] - df[pax_l24]).clip(lower=0),
            np.nan,
        )

    df["MONTH_SIN"] = np.sin(2 * np.pi * df["DEP_MONTH"] / 12.0)
    df["MONTH_COS"] = np.cos(2 * np.pi * df["DEP_MONTH"] / 12.0)
    df["SUMMER"] = df["DEP_MONTH"].isin([6, 7, 8]).astype(int)
    df["WINTER"] = df["DEP_MONTH"].isin([12, 1]).astype(int)

    if "ORG_CARGO_WGT" in df.columns and "DES_CARGO_WGT" in df.columns:
        df["COMB_CARGO_WGT"] = (
            np.log1p(df["ORG_CARGO_WGT"].clip(lower=0))
            + np.log1p(df["DES_CARGO_WGT"].clip(lower=0))
        )

    if "ORG_ENPLANEMENT" in df.columns and "DES_ENPLANEMENT" in df.columns:
        df["COMB_ENP_LOG"] = (
            np.log1p(df["ORG_ENPLANEMENT"].clip(lower=0))
            + np.log1p(df["DES_ENPLANEMENT"].clip(lower=0))
        )

    if "Route_Distance_Miles" in df.columns:
        df["is_long_haul"] = (df["Route_Distance_Miles"] >= 3500).astype(int)

    return df


full_spine = engineer_final_features(full_spine)


In [40]:
pd.set_option('display.max_columns', 100)
full_spine.columns

Index(['ORG_AIRPRT', 'DES_AIRPRT', 'DEP_YEAR', 'DEP_MONTH', 'ROUTE',
       'OD_REV_USD_LOG', 'OD_PAX_LOG', 'WEEKEND', 'HOLIDAY', 'EASTER', 'COVID',
       'MPAX_L24', 'PAX_SHARE_L24', 'AIRLN_COUNT_L24', 'ORG_CARGO_WGT',
       'ORG_CARGO_SL', 'ORG_CARGO_HUB', 'DES_CARGO_WGT', 'DES_CARGO_SL',
       'DES_CARGO_HUB', 'ORG_ENP_SL', 'ORG_ENP_HUB', 'ORG_ENPLANEMENT',
       'DES_ENP_SL', 'DES_ENP_HUB', 'DES_ENPLANEMENT', 'OD_PAX_LOG_L24',
       'OD_REV_USD_LOG_L24', 'L24_COVID', 'ORG_CNTRY_CD',
       'ORG_LNGST_RUNWAY_FT_QTY', 'ORG_ELEVATN_FT_QTY', 'ORG_CNTY_FIPS',
       'ORG_CBSA_CD', 'DES_CNTRY_CD', 'DES_LNGST_RUNWAY_FT_QTY',
       'DES_ELEVATN_FT_QTY', 'DES_CNTY_FIPS', 'DES_CBSA_CD',
       'ORG_CNTY_GDP_LOG', 'ORG_CNTY_INC', 'DES_CNTY_GDP_LOG', 'DES_CNTY_INC',
       'ORG_CNTRY_GDP', 'DES_CNTRY_GDP', 'ORG_CBSA_POP', 'ORG_CBSA_POP_GRWT',
       'ORG_CBSA_NET_MIG', 'DES_CBSA_POP', 'DES_CBSA_POP_GRWT',
       'DES_CBSA_NET_MIG', 'MIG_OUT_LOG', 'MIG_IN_LOG', 'MIG_GROSS_LOG',
       'MI

In [ ]:
full_spine['ORG_CARGO_WGT_LOG'] = np.log1p(full_spine['ORG_CARGO_WGT'])
full_spine['DES_CARGO_WGT_LOG'] = np.log1p(full_spine['DES_CARGO_WGT'])
full_spine['ORG_ENP_LOG'] = np.log1p(full_spine['ORG_ENPLANEMENT'])
full_spine['DES_ENP_LOG'] = np.log1p(full_spine['DES_ENPLANEMENT'])
full_spine.ORG_ENPLANEMENT.describe()

count    1,508,668.00
mean     8,434,044.86
std     10,152,226.56
min        232,325.00
25%      1,729,395.00
50%      3,791,807.00
75%     11,615,954.00
max     53,505,795.00
Name: ORG_ENPLANEMENT, dtype: float64

In [ ]:
# Drop the columns that might cause data leakage or is obviously correlated
cols_to_drop = [
    "Frozen_Route_Month_Pax_log1p",
    "Frozen_Route_Month_Rev_log1p",
    "Route_Active_Months_Hist",
    "ORG_CARGO_WGT",
    "DES_CARGO_WGT",
    "ORG_ENPLANEMENT",
    "DES_ENPLANEMENT",
    "ORG_LAT",
    "ORG_LNGTD",
    "DES_LAT",
    "DES_LNGTD",
    "ORG_CNTY_FIPS",
    "DES_CNTY_FIPS",
    "ORG_CBSA_CD",
    "DES_CBSA_CD",
]

full_spine = full_spine.drop(columns=[c for c in cols_to_drop if c in full_spine.columns])

# Define data leak free feature list X 
non_feature_cols = {
    "ROUTE", "ORG_AIRPRT", "DES_AIRPRT", "DEP_YEAR", "DEP_MONTH", "_date",
    "OD_PAX_LOG", "OD_REV_USD_LOG",  
    "COVID",        
}

feature_cols = [c for c in full_spine.columns if c not in non_feature_cols]
print(f"Clean, leak-free feature count: {len(feature_cols)}")
print(feature_cols)


Clean, leak-free feature count: 53
['WEEKEND', 'HOLIDAY', 'EASTER', 'MPAX_L24', 'PAX_SHARE_L24', 'AIRLN_COUNT_L24', 'ORG_CARGO_SL', 'ORG_CARGO_HUB', 'DES_CARGO_SL', 'DES_CARGO_HUB', 'ORG_ENP_SL', 'ORG_ENP_HUB', 'DES_ENP_SL', 'DES_ENP_HUB', 'OD_PAX_LOG_L24', 'OD_REV_USD_LOG_L24', 'L24_COVID', 'ORG_CNTRY_CD', 'ORG_LNGST_RUNWAY_FT_QTY', 'ORG_ELEVATN_FT_QTY', 'DES_CNTRY_CD', 'DES_LNGST_RUNWAY_FT_QTY', 'DES_ELEVATN_FT_QTY', 'ORG_CNTY_GDP_LOG', 'ORG_CNTY_INC', 'DES_CNTY_GDP_LOG', 'DES_CNTY_INC', 'ORG_CNTRY_GDP', 'DES_CNTRY_GDP', 'ORG_CBSA_POP', 'ORG_CBSA_POP_GRWT', 'ORG_CBSA_NET_MIG', 'DES_CBSA_POP', 'DES_CBSA_POP_GRWT', 'DES_CBSA_NET_MIG', 'MIG_OUT_LOG', 'MIG_IN_LOG', 'MIG_GROSS_LOG', 'MIG_NET_LOG', 'DIST_LOG', 'ORG_CNTY_INC_LOG', 'DES_CNTY_INC_LOG', 'INTRNAT', 'ECON_LOG', 'ECON_SUM_LOG', 'PURCHASE_PWR_LOG', 'AVG_FARE_LOG_L24', 'MONTH_SIN', 'MONTH_COS', 'SUMMER', 'WINTER', 'COMB_CARGO_WGT', 'COMB_ENP_LOG']


In [51]:
full_spine.columns

Index(['ORG_AIRPRT', 'DES_AIRPRT', 'DEP_YEAR', 'DEP_MONTH', 'ROUTE',
       'OD_REV_USD_LOG', 'OD_PAX_LOG', 'WEEKEND', 'HOLIDAY', 'EASTER', 'COVID',
       'MPAX_L24', 'PAX_SHARE_L24', 'AIRLN_COUNT_L24', 'ORG_CARGO_SL',
       'ORG_CARGO_HUB', 'DES_CARGO_SL', 'DES_CARGO_HUB', 'ORG_ENP_SL',
       'ORG_ENP_HUB', 'DES_ENP_SL', 'DES_ENP_HUB', 'OD_PAX_LOG_L24',
       'OD_REV_USD_LOG_L24', 'L24_COVID', 'ORG_CNTRY_CD',
       'ORG_LNGST_RUNWAY_FT_QTY', 'ORG_ELEVATN_FT_QTY', 'DES_CNTRY_CD',
       'DES_LNGST_RUNWAY_FT_QTY', 'DES_ELEVATN_FT_QTY', 'ORG_CNTY_GDP_LOG',
       'ORG_CNTY_INC', 'DES_CNTY_GDP_LOG', 'DES_CNTY_INC', 'ORG_CNTRY_GDP',
       'DES_CNTRY_GDP', 'ORG_CBSA_POP', 'ORG_CBSA_POP_GRWT',
       'ORG_CBSA_NET_MIG', 'DES_CBSA_POP', 'DES_CBSA_POP_GRWT',
       'DES_CBSA_NET_MIG', 'MIG_OUT_LOG', 'MIG_IN_LOG', 'MIG_GROSS_LOG',
       'MIG_NET_LOG', 'DIST_LOG', 'ORG_CNTY_INC_LOG', 'DES_CNTY_INC_LOG',
       'INTRNAT', 'ECON_LOG', 'ECON_SUM_LOG', 'PURCHASE_PWR_LOG',
       'AVG_FARE_LOG

In [52]:
full_spine.to_csv(DATA_DIR/'made/AA_MASTER_V10.csv',index=False)